# Create Wellcome Trust DBT India Alliance Awards

Creates India Alliance (Hyderabad; Wellcome Trust + Department of Biotechnology, India)
fellowships and grants from the alliance's own grant list deposited with Europe PMC,
served by the GRIST grants API under the funder name "DBT/Wellcome Trust India Alliance"
(FundRef 10.13039/501100009053). **543 awards, 2009-2024, 100% title/PI/host
institution, 99.8% dates, 98.5% amount (INR 17.7B), 84% abstract** (local run
2026-10-01, oxjob #1451). Template: CreateDiabetesUKAwards (same GRIST source shape).

Types: Intermediate / Early Career / Senior Fellowships (basic biomedical), Clinical &
Public Health (CPH) Early Career / Intermediate / Senior Fellowships, Research Training
Fellowships, Margdarshi Fellowships (`funding_type` fellowship, 511); Team Science Grants
and Clinical/Public Health Research Centres (`research`, 32). Nothing filtered.

**Overlap with the Wellcome ingest (checked before shipping):** CreateWellcomeAwards
(Wellcome 360Giving, F4320311904) holds only the three GBP 40M block grants Wellcome made
TO India Alliance (109504/Z/15/Z, 109508/Z/15/Z, 218696/Z/19/Z) and none of these 543
individual awards (0 of the 44 Wellcome-style refs below are in wellcome_raw). No
duplication; the verification cell `in_wellcome_ingest` must stay 0.

**Prerequisites:** run `scripts/local/india_alliance_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/india_alliance/india_alliance_projects.parquet`
(§1.4 shrink guard; GRIST paging is unstable, so the script unions query variants until
it has every record: 544/544 on 2026-10-01, collapsed to 543 grants).

**`funder_award_id` (§2.1.1):** the India Alliance reference as grantees cite it:
`IA/{I|E|S|CPHE|CPHI|CPHS|TSG|...}/YY/N/NNNNNN` (499 awards; 14 of them carry irregular
forms the alliance itself issued, e.g. `IA/I/11/2500278`, `IA/S(I)/14/1/501286`, and
citations use those same strings, so they ship verbatim) or, for the first 2009-2012
fellowships, the Wellcome-style reference `500164/Z/09/Z` (44 awards; GRIST stores these
with hyphens, `500164-Z-09-Z`, so the script ships the slash form). 388 of the 713
distinct stub ids under F4320325580 collapse onto this ingest.

**Citation split (not fixable here):** the same India Alliance references are also cited
under other funder ids, where they cannot collapse (award ids hash funder_id):
Wellcome Trust F4320311904 (83 IA/ refs + 29 Wellcome-style refs match these awards) and
DBT India F4320320717 (48 + 3). Those stay as Wellcome/DBT acknowledgement shells.

`description` = the scientific abstract. `lead_investigator` = the fellow / grant holder
with host institution (+ ROR asserted by the funder, 72%); ORCID where GRIST has it (3%).
Amounts are INR as deposited.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320325580`
- display_name: The Wellcome Trust DBT India Alliance
- ror_id / doi: from `openalex.funders.funders` (ror 04reqzt68, doi 10.13039/501100009053)

**Priority:** `543` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.india_alliance_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/india_alliance/india_alliance_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.india_alliance_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.india_alliance_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.india_alliance_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320325580 is a Crossref-registered F4320* funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320325580;

## Step 2: Create The Wellcome Trust DBT India Alliance Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.india_alliance_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320325580  -- The Wellcome Trust DBT India Alliance
)
,

people AS (
    -- Grant holders in source order -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(
                from_json(people_json, 'ARRAY<STRUCT<given_name:STRING, family_name:STRING, title:STRING, orcid:STRING, institution:STRING, institution_ror:STRING>>'),
                p -> p.family_name IS NOT NULL
            ),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]$' THEN CONCAT('https://orcid.org/', p.orcid) END,
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CASE WHEN p.institution_ror IS NOT NULL
                                THEN array(named_struct('id', p.institution_ror, 'type', 'ror', 'asserted_by', 'funder'))
                                ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) END
                )
            )
        ) AS investigators
    FROM openalex.awards.india_alliance_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.abstract), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN NULLIF(TRIM(g.currency), '') END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.award_type), '') as funder_scheme,
    'india_alliance_europepmc_grist' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.india_alliance_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'india_alliance_europepmc_grist' AND priority = 543;

-- Insert into openalex_awards_raw with priority.
-- Priority 543: direct-from-funder ingest (higher wins under the 2026-06-20 DESC
-- dedup, oxjob #500), so it outranks the priority-0 acknowledgement shells that
-- share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    543 as priority
FROM openalex.awards.india_alliance_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT(DISTINCT LOWER(funder_award_id)) as distinct_award_ids
FROM openalex.awards.india_alliance_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.india_alliance_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage
SELECT
    COUNT(*) as total,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amt, MAX(amount) as max_amt, ROUND(AVG(amount), 0) as avg_amt,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) as pct_start_year,
    ROUND(COUNT(lead_investigator.family_name) * 100.0 / COUNT(*), 1) as pct_pi,
    ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) as pct_institution
FROM openalex.awards.india_alliance_awards;

In [ ]:
%sql
SELECT funder_scheme, currency, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.india_alliance_awards
GROUP BY funder_scheme, currency ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.india_alliance_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.india_alliance_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.india_alliance_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.5 funder consistency
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.india_alliance_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- Award-id shape (2.1.1): IA/... references or slash-form Wellcome-style refs; no hyphen form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^IA/[A-Z]+/[0-9]{2}/[0-9]+/[0-9]+$' THEN 1 ELSE 0 END) AS ia_ref,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{6}/Z/[0-9]{2}/Z$' THEN 1 ELSE 0 END) AS wellcome_style_ref,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{6}-Z-' THEN 1 ELSE 0 END) AS hyphen_form, -- must be 0
    SUM(CASE WHEN NOT (funder_award_id RLIKE '^IA/[A-Z]+/[0-9]{2}/[0-9]+/[0-9]+$' OR funder_award_id RLIKE '^[0-9]{6}/Z/[0-9]{2}/Z$') THEN 1 ELSE 0 END) AS other_shape, -- expect 14 funder-issued irregular IA refs
    SUM(CASE WHEN NOT (funder_award_id LIKE 'IA/%' OR funder_award_id RLIKE '^[0-9]{6}/Z/[0-9]{2}/Z$') THEN 1 ELSE 0 END) AS not_ia_at_all -- must be 0
FROM openalex.awards.india_alliance_awards;

In [ ]:
%sql
-- Existing acknowledgement stubs that collapse onto this ingest (expect ~388 distinct ids).
SELECT c.provenance, COUNT(*) AS stub_rows, COUNT(s.id) AS collapsed
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.india_alliance_awards s ON s.id = c.id
WHERE c.funder_id = 4320325580 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Overlap guard: none of these awards may already sit in the Wellcome direct ingest.
SELECT COUNT(*) AS in_wellcome_ingest  -- expect 0
FROM openalex.awards.india_alliance_awards a
JOIN openalex.awards.wellcome_awards w ON LOWER(w.funder_award_id) = LOWER(a.funder_award_id);

In [ ]:
%sql
-- Section 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'india_alliance_europepmc_grist'
GROUP BY provenance, priority;